# 基于 OpenVINO 的端侧现场交付状态自动说明助手

本 Notebook 是魔搭灵感流提交入口，目标是跑通一个真实可演示的端侧多模态应用：

```text
交付现场图片
  -> 下载并加载 OpenVINO VLM 模型
  -> 实时识别物品、位置、参照物、交付状态
  -> 根据必要上下文生成收件人通知
  -> 生成 TTS 播报文本
  -> 生成 Markdown 留档记录
  -> 支持二次追问
```

场景覆盖：企业园区、医院科室、工厂仓库。


## 1. 环境安装

参考官方 baseline：`https://github.com/openvino-dev-samples/modelscope-workshop`。

本 Notebook 重点使用 Lab 1 的 Qwen3-VL OpenVINO 模型；ASR/TTS 后续可继续接入官方 Lab 2 / Lab 3。


In [ ]:
# 魔搭灵感流环境中建议先执行。
# 如果依赖已经安装，可以跳过这一格。
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt


## 2. 导入模块与读取样例

`samples/manifest.example.json` 只保留必要信息：图片、场景、收件对象、物品提示、补充备注和追问。

最终通知和 TTS 文本由模型与代码实时生成，不在样例中预置。


In [ ]:
from pathlib import Path
import json

from IPython.display import Image, Markdown, display

from src.delivery_generator import generate_delivery_result
from src.delivery_schema import VisionObservation
from src.openvino_vlm_service import (
    DEFAULT_VLM_MODEL_DIR,
    DEFAULT_VLM_MODEL_ID,
    OpenVINODeliveryVLM,
    download_vlm_model,
)
from src.sample_loader import context_from_sample, load_manifest

project_root = Path.cwd()
samples = load_manifest("samples/manifest.example.json")
print("样例数量:", len(samples))
for sample in samples:
    print(sample["id"], sample["scene"], sample["image_path"])


## 3. 查看三张示例图片

这一步用于确认图片文件路径正确，也方便在文章中截图展示。


In [ ]:
for sample in samples:
    print("
==", sample["id"], "==")
    print(sample["image_description"])
    display(Image(filename=sample["image_path"], width=420))


## 4. 下载并加载 OpenVINO VLM 模型

默认模型：`snake7gun/Qwen3-VL-4B-Instruct-int4-ov`。

首次下载会比较久。模型会保存到 `models/Qwen3-VL-4B-Instruct-int4-ov`。


In [ ]:
RUN_REAL_VLM = True
DEVICE = "AUTO"
MODEL_ID = DEFAULT_VLM_MODEL_ID
MODEL_DIR = DEFAULT_VLM_MODEL_DIR

vlm_service = None
if RUN_REAL_VLM:
    try:
        model_dir = download_vlm_model(model_id=MODEL_ID, local_dir=MODEL_DIR)
        print("模型目录:", model_dir)
        vlm_service = OpenVINODeliveryVLM(model_dir=model_dir, device=DEVICE)
        print("OpenVINO VLM 加载成功")
    except Exception as exc:
        print("OpenVINO VLM 加载失败，将使用降级演示路径。")
        print(type(exc).__name__, exc)
        vlm_service = None


## 5. 定义降级观察逻辑

真实提交时应使用上一步加载的 VLM 服务。

如果当前环境暂时无法安装或加载模型，下面的降级逻辑只用于验证后处理链路，不作为最终评测结果。


In [ ]:
def fallback_observation(sample):
    return VisionObservation(
        item=sample.get("delivery_context", {}).get("item_hint", "待确认物品"),
        location="请以图片识别结果为准",
        landmark=sample.get("image_description", "暂无明显参照物"),
        status="已拍照记录，等待真实 VLM 识别确认",
        confidence_note="当前使用图片描述降级演示，正式运行请启用 OpenVINO VLM。",
        raw_description=sample.get("image_description", ""),
    )


## 6. 对示例图片实时推理并生成交付说明

每张图的处理流程：

```text
图片 -> VLM 结构化观察 -> 场景模板生成通知 -> TTS 文本 -> Markdown 留档
```


In [ ]:
outputs = []
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

for sample in samples:
    print("
" + "=" * 80)
    print("样例:", sample["id"])
    display(Image(filename=sample["image_path"], width=420))

    context = context_from_sample(sample)
    followup_question = ""
    if sample.get("followup_cases"):
        followup_question = sample["followup_cases"][0].get("question", "")

    if vlm_service is not None:
        observation = vlm_service.observe_delivery_image(sample["image_path"])
    else:
        observation = fallback_observation(sample)

    result = generate_delivery_result(
        context=context,
        observation=observation,
        followup_question=followup_question,
    )

    record = {
        "sample_id": sample["id"],
        "image_path": sample["image_path"],
        "observation": observation.__dict__,
        "result": result.to_dict(),
    }
    outputs.append(record)

    print("【VLM 结构化观察】")
    print(json.dumps(observation.__dict__, ensure_ascii=False, indent=2))
    print("
【收件人通知】")
    print(result.recipient_message)
    print("
【TTS 播报文本】")
    print(result.tts_text)
    if result.followup_answer:
        print("
【二次追问】", followup_question)
        print(result.followup_answer)

    (output_dir / f"{sample['id']}_record.md").write_text(result.archive_markdown, encoding="utf-8")

(output_dir / "delivery_results.json").write_text(
    json.dumps(outputs, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("
已写入 outputs/delivery_results.json 和各样例 Markdown 留档。")


## 7. 查看生成的 Markdown 留档

以下内容可直接作为交付记录，也可以粘贴到研习社文章中展示。


In [ ]:
for path in sorted(Path("outputs").glob("*_record.md")):
    print("
---", path, "---")
    display(Markdown(path.read_text(encoding="utf-8")))


## 8. 可选：启动 Gradio 服务

运行后可以在浏览器中上传任意交付现场图片，实时生成送达说明。

在魔搭环境中，如果端口不可访问，可只运行前面的 Notebook 单元。


In [ ]:
# from src.gradio_service import build_demo
# demo = build_demo()
# demo.launch(server_name="0.0.0.0", server_port=7860, share=False)


## 9. 可选：TTS 音频合成接入点

当前项目已经生成 `tts_text`。如果要进一步生成音频，可参考官方 `lab3-text-to-speech`，将 `result.tts_text` 传入 TTS helper。

比赛文章中可以先展示 TTS 播报文本；如果灵感流环境跑通 TTS，再补充音频文件或截图。


In [ ]:
# 示例伪代码，请按官方 lab3-text-to-speech 的 helper 实际函数名调整：
# from qwen_3_tts_helper import OVQwen3TTSModel
# tts_model = OVQwen3TTSModel.from_pretrained(
#     model_dir="lab3-text-to-speech/Qwen3-TTS-0.6B-fp16-ov",
#     device="AUTO",
# )
# for item in outputs:
#     text = item["result"]["tts_text"]
#     audio = tts_model.synthesize(text)
#     audio.save(f"outputs/{item['sample_id']}_tts.wav")


## 10. 总结

本 Notebook 已完成比赛所需的核心闭环：

- 下载并加载 OpenVINO VLM 模型
- 使用三张示例交付图片实时推理
- 根据必要上下文生成交付通知
- 生成 TTS 播报文本
- 支持二次追问
- 导出 Markdown 和 JSON 结果

后续可在研习社文章中补充运行截图、GitHub 仓库地址和 Copaw Skill 运行截图。
